In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [2]:
import numpy as np
from src.data_loader import load_raw_data
from src.preprocessing import clean_raw_data, chronological_split, fit_scaler, apply_scaler, inverse_transform_column
from src.feature_engineering import build_features
from src.sequence_builder import create_sequences_for_all_splits
from src.evaluation.metrics import evaluate_all
from src.models.baseline import naive_forecast, moving_average_forecast, LinearRegressionForecaster
from src.config import MODEL_FEATURE_COLUMNS, TARGET_COL, DEFAULT_LOOKBACK, FORECAST_HORIZON

df_raw = load_raw_data()
df_clean = clean_raw_data(df_raw)
df_features = build_features(df_clean)
splits = chronological_split(df_features)

scale_columns = MODEL_FEATURE_COLUMNS + [TARGET_COL]
scaler = fit_scaler(splits.train, scale_columns)

train_scaled = apply_scaler(splits.train, scaler, scale_columns)
val_scaled = apply_scaler(splits.val, scaler, scale_columns)
test_scaled = apply_scaler(splits.test, scaler, scale_columns)

sequences = create_sequences_for_all_splits(
    train_scaled, val_scaled, test_scaled, MODEL_FEATURE_COLUMNS
)
X_train, y_train_scaled = sequences["train"]
X_val, y_val_scaled = sequences["val"]
X_test, y_test_scaled = sequences["test"]

print("Ready:", X_train.shape, X_val.shape, X_test.shape)

2026-09-10 17:38:40 | INFO     | src.data_loader | Loading raw dataset from C:\Users\Ram\OneDrive\Desktop\energy-demand-lstm\data\raw\continuous_dataset.csv
2026-09-10 17:38:41 | INFO     | src.data_loader | Loaded raw dataset: 48048 rows, 17 columns
2026-09-10 17:38:41 | INFO     | src.preprocessing | Cleaning complete: 48048 rows, 4 flagged as diagnostic extreme events (|z| > 4.0)
2026-09-10 17:38:41 | INFO     | src.feature_engineering | Added national weather averages: temp_avg, humidity_avg, precip_avg, wind_avg
2026-09-10 17:38:41 | INFO     | src.feature_engineering | Added calendar features and cyclical encodings
2026-09-10 17:38:41 | INFO     | src.feature_engineering | Added lag features: ['lag_1', 'lag_24', 'lag_48', 'lag_168']
2026-09-10 17:38:41 | INFO     | src.feature_engineering | Added rolling features: rolling_mean_24, rolling_std_24, rolling_mean_168
2026-09-10 17:38:41 | INFO     | src.feature_engineering | Feature engineering complete: 48048 rows before, 47880 rows

In [3]:
y_true_naive, y_pred_naive = naive_forecast(splits.test)
naive_metrics = evaluate_all(y_true_naive, y_pred_naive)
print("Naive baseline:", naive_metrics)

Naive baseline: {'MAE': 165.8332977294922, 'RMSE': 213.03768920898438, 'MAPE': 13.862836360931396, 'sMAPE': 13.6790931224823, 'R2': -0.3298739194869995}


In [4]:
y_true_ma, y_pred_ma = moving_average_forecast(splits.test)
ma_metrics = evaluate_all(y_true_ma, y_pred_ma)
print("Moving-average baseline:", ma_metrics)

Moving-average baseline: {'MAE': 132.92239379882812, 'RMSE': 165.12034606933594, 'MAPE': 11.152240633964539, 'sMAPE': 10.994116961956024, 'R2': 0.20108836889266968}


In [5]:
lr_model = LinearRegressionForecaster()
lr_model.fit(X_train, y_train_scaled)

y_pred_test_scaled = lr_model.predict(X_test)

y_true_lr = inverse_transform_column(y_test_scaled, scaler, scale_columns, TARGET_COL)
y_pred_lr = inverse_transform_column(y_pred_test_scaled, scaler, scale_columns, TARGET_COL)

lr_metrics = evaluate_all(y_true_lr, y_pred_lr)
print("Linear Regression baseline:", lr_metrics)

2026-09-10 17:38:56 | INFO     | src.models.baseline | LinearRegressionForecaster fit on 33325 samples, 20 features (last timestep only)
Linear Regression baseline: {'MAE': 92.6028062600483, 'RMSE': 117.55615183031193, 'MAPE': 7.920299034299069, 'sMAPE': 7.806543169949691, 'R2': 0.5950621759598956}


In [7]:
import pandas as pd

results = pd.DataFrame({
    "Naive": naive_metrics,
    "Moving Average": ma_metrics,
    "Linear Regression": lr_metrics,
}).T
print(results)

                          MAE        RMSE       MAPE      sMAPE        R2
Naive              165.833298  213.037689  13.862836  13.679093 -0.329874
Moving Average     132.922394  165.120346  11.152241  10.994117  0.201088
Linear Regression   92.602806  117.556152   7.920299   7.806543  0.595062


In [8]:
import json
from pathlib import Path

experiments_path = Path.cwd().parent / "experiments" / "model_comparison.json"
experiments_path.parent.mkdir(exist_ok=True)

all_results = {
    "Naive": naive_metrics,
    "Moving Average": ma_metrics,
    "Linear Regression": lr_metrics,
}

with open(experiments_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Saved results to {experiments_path}")

Saved results to c:\Users\Ram\OneDrive\Desktop\energy-demand-lstm\experiments\model_comparison.json
